# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mukeshboolani786/flyrank-internship-ml/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Capstone — Refresh / Content Opportunity Scoring
Ranked review queue for content pages: which pages should a human look at first?
Lane: Refresh / Content Opportunity Scoring Data: FlyRank/internship-warehouse (Hugging Face, gated) — fact_content_daily_performance, dim_clients, dim_content Output: a ranked, reason-coded action queue — not a claim about Google's algorithm.

This notebook queries the warehouse directly with DuckDB over hf:// Parquet (no full download), builds leakage-safe historical features, defines a future-looking opportunity label from the data's own distribution, trains a model against a transparent rule baseline on the same time-aware split, and produces the final ranked recommendations with reason codes that back the deployed paper.

Run top to bottom in Colab with the internship-token secret set. Each heavy DuckDB scan runs once and caches to work/outputs/ — reruns after that are fast.

In [2]:
%pip install -q duckdb pandas numpy scikit-learn matplotlib huggingface_hub

In [3]:
# Authenticate — token stays in Colab Secrets, never in a cell or printed.
from google.colab import userdata
import os

HF_TOKEN = userdata.get("internship-token")
os.environ["HF_TOKEN"] = HF_TOKEN  # lets duckdb's httpfs pick it up too
print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [4]:
import duckdb

con = duckdb.connect()
con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{}'
    )
""".format(HF_TOKEN))

# Query hf:// paths directly — no snapshot_download, no full warehouse copy.
REPO = "hf://datasets/FlyRank/internship-warehouse"
DIM_CLIENTS = f"read_parquet('{REPO}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REPO}/dim_content.parquet')"

# Near-free sanity check: metadata only, not a full scan.
counts = con.sql(f"""
    SELECT
        (SELECT COUNT(*) FROM {DIM_CLIENTS})  AS n_clients,
        (SELECT COUNT(*) FROM {DIM_CONTENT})  AS n_content
""").df()
counts

,n_clients,n_content
0,104,519606


## 1. Question

*The research question and the decision it supports.*

1. Question
Research question:

Using a content page's search and engagement history available as of a cutoff date, can pages be ranked so that the pages most likely to see a meaningful future decline in search demand cluster at the top of the list — producing a review queue that beats a simple momentum rule?

The decision this supports: an editorial/SEO reviewer has limited time and cannot manually audit every page every month. The queue tells them which pages to look at first for a refresh, and why (reason codes) — not whether a refresh will work, and not what Google's algorithm is doing.

What the output is, precisely: a ranked list of anonymized content_hash_id values with a score, an action (REFRESH_REVIEW, MONITOR, PROTECT, INVESTIGATE, RECOVERY_REVIEW), and reason codes built from the same features the model saw. It is decision-support for a human reviewer, nothing more.

What it explicitly does not claim:

it does not predict search-engine ranking behavior;
it does not claim a refresh will cause traffic to recover;
"decline" is an operational label defined below from observed history, not a measure of content quality.

In [5]:
# No computation needed for this section — the question is fixed above.
# (Kept as an empty-but-present code cell so the notebook structure matches the assignment skeleton.)
print("Research question set. Proceeding to Data.")

Research question set. Proceeding to Data.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*


Release: FlyRank/internship-warehouse (build v20260703), Hugging Face, gated dataset — instant approval, read-only token via Colab Secrets.

Confirmed against the live table: fact_content_daily_performance has 78,835,655 rows spanning 2025-01-27 to 2026-06-30 — matches the published release numbers exactly (metadata-only check, near-free).

Tables used:

Table	Grain	Use here
dim_clients	one row per pseudonymized client	per-client history start (gsc_data_start, ga4_data_start) so windows never predate a client's own tracking
fact_content_daily_performance	intended grain: one row per (content_hash_id, report_date)	the time series — features and label both come from here
dim_content and fact_content_query_90d are not used: dim_content carries no columns this schema needs beyond the ID, and fact_content_query_90d's fixed 90-day window overlaps the label period for any recent-month label — pulling it in would risk exactly the leakage this design avoids.

Columns actually used (confirmed against the live schema — no assumed columns like content_type/word_count/days_since_update, which do not exist in this table): report_date, client_hash_id, content_hash_id, gsc_data_available, ga4_data_available, gsc_impressions, gsc_clicks, gsc_avg_position, ga4_pageviews, sessions_organic, ga4_engaged_sessions, ga4_total_engagement_sec, scroll_events, month.

The grain probe — what we actually found (correcting the initial assumption): we probed a mid-panel month (2026-01), not the sealed final month, per the data skill's iteration rule. GROUP BY content_hash_id, report_date HAVING COUNT(*) > 1 returned an empty result for that month — no exact-duplicate page-days there. The originally-suspected example (content_b11af2c267f7170f, 2026-06-19) falls inside the sealed _sample month, which we deliberately do not open while building label logic, so we cannot confirm that specific row here. We keep the defensive GROUP BY content_hash_id, report_date + ANY_VALUE() deduplication in every query regardless — it's documented elsewhere in the panel, it's free when no duplicates exist, and it protects against silent double-counting if a different month does carry them. This is the honest result: the probe we could safely run came back clean; we don't claim more than that.

Per-client history coverage (from dim_clients, 104 rows): only 67 of 104 clients carry a non-null gsc_data_start, and only 51 of 104 carry a non-null ga4_data_start — the remaining clients have no confirmed tracking-start date on either signal. Our eligibility join (gsc_data_start <= feat_start) correctly drops the null-start clients, since a NULL comparison is neither true nor false in SQL — they are excluded, not silently treated as "always eligible." gsc_data_start ranges from 2025-01-27 to 2026-06-02 across clients with a known start (median 2025-11-05) — confirming the panel really is unbalanced, exactly as documented, and windows must be checked per-client rather than assumed globally.

Public-safety: everything below stays at anonymized content_hash_id / client_hash_id grain. No URLs, client names, or raw queries ever leave the warehouse — this lane never touches fact_content_query_90d, the only table with anything query-shaped in it.

In [6]:
# --- Metadata-only checks: near-free, confirm we're pointed at the right table/date range ---
FACT_ALL = f"read_parquet('{REPO}/fact_content_daily_performance/*/*.parquet')"

meta = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {FACT_ALL}
""").df()
meta

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,min_date,max_date
0,78835655,2025-01-27,2026-06-30


In [7]:
# --- Grain probe on ONE mid-panel month first (cheap) — confirm duplicate page-days exist,
# then confirm they are EXACT duplicates before deciding to dedupe.
FACT_PROBE_MONTH = f"read_parquet('{REPO}/fact_content_daily_performance/month=2026-01/*.parquet')"

dupe_probe = con.sql(f"""
    SELECT content_hash_id, report_date, COUNT(*) AS n
    FROM {FACT_PROBE_MONTH}
    GROUP BY content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
dupe_probe

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,report_date,n


In [8]:
# For one duplicated (content_hash_id, report_date) pair, confirm every column has nunique()==1
# i.e. the "duplicate" rows are exact copies, not distinct observations.
if len(dupe_probe):
    cid, rdate = dupe_probe.iloc[0]["content_hash_id"], dupe_probe.iloc[0]["report_date"]
    dup_rows = con.sql(f"""
        SELECT * FROM {FACT_PROBE_MONTH}
        WHERE content_hash_id = '{cid}' AND report_date = DATE '{rdate}'
    """).df()
    print("Rows found for this page-day:", len(dup_rows))
    print("Columns where the duplicate rows actually differ (should be EMPTY):")
    print([c for c in dup_rows.columns if dup_rows[c].nunique(dropna=False) > 1])
else:
    print("No duplicates found in this month's probe — will still dedupe defensively downstream.")

No duplicates found in this month's probe — will still dedupe defensively downstream.


In [9]:
# --- Per-client history coverage: define windows relative to each client's OWN start date,
# never a single global calendar window (unbalanced panel warning from the data skill).
clients = con.sql(f"""
    SELECT client_hash_id, gsc_data_start, ga4_data_start
    FROM {DIM_CLIENTS}
""").df()
clients.describe(include="all")

,client_hash_id,gsc_data_start,ga4_data_start
count,104,67,51
unique,104,NaN,NaN
top,client_04660893ae39614a,NaN,NaN
freq,1,NaN,NaN
mean,NaN,2025-11-17 00:42:59.104477,2026-02-23 05:38:49.411764
min,NaN,2025-01-27 00:00:00,2025-10-29 00:00:00
25%,NaN,2025-09-24 00:00:00,2026-02-19 00:00:00
50%,NaN,2025-11-05 00:00:00,2026-02-20 00:00:00
75%,NaN,2026-02-19 00:00:00,2026-03-21 12:00:00
max,NaN,2026-06-02 00:00:00,2026-06-01 00:00:00


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*



Unit of analysis
Raw grain is content page × day. The modeling unit is content page × cutoff date: for a chosen cutoff T, we build historical features from data strictly at or before T, and a label from data strictly after T. Using several cutoff dates per page (a small rolling-origin design) gives more training rows without inventing data, and keeps every row's features/label pair separated in time — the same page appearing at two cutoffs is not leakage, it's a legitimate repeated observation, similar to any panel-forecasting setup.

Cutoffs chosen
Cutoff (T)	Feature window	Label window	Role
2025-09-30	2025-08-05 → 2025-09-30 (56d)	2025-10-01 → 2025-10-28 (28d)	train
2025-11-30	2025-10-06 → 2025-11-30 (56d)	2025-12-01 → 2025-12-28 (28d)	train
2026-01-31	2025-12-07 → 2026-01-31 (56d)	2026-02-01 → 2026-02-28 (28d)	test (held out in time)
All three cutoffs and their label windows end well before 2026-06 (the sealed _sample month), and before the query table's fixed 90-day window — so no table used here overlaps the sealed test period. The 56-day feature window is split into two 28-day sub-windows (recent_28, prior_28) to build momentum features without needing more history than the shortest reasonably-covered clients have.

Eligibility (checked in code, not assumed)
A (page, cutoff) row is eligible only if:

the page's client's gsc_data_start is at least 55 days before T (full feature history exists);
gsc_data_available IS TRUE for the row-days used (three-valued flag — IS TRUE, never = TRUE, to avoid silently mishandling the documented NULL rows);
the page had non-trivial visibility in recent_28 (a minimum impressions floor, set from the data's own distribution below) — a page nobody ever saw isn't a "declining" page, it's an absent one.
Features (all computable at or before T — no future information)
For each of recent_28 and prior_28: summed gsc_impressions, gsc_clicks, sessions_organic, ga4_pageviews, ga4_engaged_sessions, ga4_total_engagement_sec, scroll_events, plus the mean of daily gsc_avg_position. From those we derive: click_change, impression_change, organic_session_change, position_change (recent − prior; positive = position number rose = worse), engagement_rate_change (engaged sessions ÷ organic sessions, recent vs prior), click_volatility (std of daily clicks across the full 56-day window), and recent_vs_baseline_ratio for clicks and impressions (recent ÷ prior, with a +1 smoothing constant so idle pages don't produce division by zero or infinite ratios).

Label — future opportunity flag
Computed from future_28, strictly after T: future_click_ratio = (future_clicks + 1) / (recent_clicks + 1). We inspect this ratio's distribution among eligible pages below and set the "meaningful decline" cutoff at its own lower quartile, rather than picking a round number first — the threshold is read off the data, not assumed. label = 1 if future_click_ratio falls at or below that quartile and the page was visible enough in recent_28 to make a decline meaningful (same visibility floor as eligibility).

Assumptions this rests on
A same-portfolio click ratio is a reasonable proxy for "page losing relevance," not a certified quality signal — it can move for reasons the dataset can't see (seasonality, a competitor's page, a SERP feature change).
Grouping by content and by time cutoff, rather than a fully random split, better mimics how this would actually be deployed (score today's pages, review next month).

In [10]:
# --- Build the month path LIST covering all cutoffs' feature + label windows (hf:// needs an
# explicit list, not a brace-glob).
MONTHS_NEEDED = ["2025-08","2025-09","2025-10","2025-11","2025-12","2026-01","2026-02"]
month_paths = [f"{REPO}/fact_content_daily_performance/month={m}/*.parquet" for m in MONTHS_NEEDED]
month_list_sql = ", ".join(f"'{p}'" for p in month_paths)

FACT_WINDOW = f"read_parquet([{month_list_sql}])"

# Grain-safe base: dedupe exact-duplicate page-days once, up front, in SQL.
BASE_CTE = f"""
base AS (
    SELECT
        content_hash_id,
        client_hash_id,
        report_date,
        ANY_VALUE(gsc_impressions)          AS gsc_impressions,
        ANY_VALUE(gsc_clicks)               AS gsc_clicks,
        ANY_VALUE(gsc_avg_position)         AS gsc_avg_position,
        ANY_VALUE(sessions_organic)         AS sessions_organic,
        ANY_VALUE(ga4_pageviews)            AS ga4_pageviews,
        ANY_VALUE(ga4_engaged_sessions)     AS ga4_engaged_sessions,
        ANY_VALUE(ga4_total_engagement_sec) AS ga4_total_engagement_sec,
        ANY_VALUE(scroll_events)            AS scroll_events,
        ANY_VALUE(gsc_data_available)       AS gsc_data_available,
        ANY_VALUE(ga4_data_available)       AS ga4_data_available
    FROM {FACT_WINDOW}
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id, report_date
)
"""
print("Base CTE ready.")

Base CTE ready.


In [11]:
def cutoff_query(cutoff, feat_start, prior_split, label_end):
    """Build the per-cutoff aggregation SQL. All dates are strings 'YYYY-MM-DD'.
    recent_28  = (prior_split, cutoff]
    prior_28   = [feat_start, prior_split]
    future_28  = (cutoff, label_end]
    """
    return f"""
    WITH {BASE_CTE}
    , eligible_clients AS (
        SELECT client_hash_id
        FROM {DIM_CLIENTS}
        WHERE gsc_data_start <= DATE '{feat_start}'
    )
    SELECT
        b.content_hash_id,
        DATE '{cutoff}' AS cutoff_date,

        SUM(CASE WHEN b.report_date > DATE '{prior_split}' AND b.report_date <= DATE '{cutoff}' THEN b.gsc_impressions END)          AS recent_impressions,
        SUM(CASE WHEN b.report_date > DATE '{prior_split}' AND b.report_date <= DATE '{cutoff}' THEN b.gsc_clicks END)               AS recent_clicks,
        AVG(CASE WHEN b.report_date > DATE '{prior_split}' AND b.report_date <= DATE '{cutoff}' THEN b.gsc_avg_position END)          AS recent_position,
        SUM(CASE WHEN b.report_date > DATE '{prior_split}' AND b.report_date <= DATE '{cutoff}' THEN b.sessions_organic END)          AS recent_organic_sessions,
        SUM(CASE WHEN b.report_date > DATE '{prior_split}' AND b.report_date <= DATE '{cutoff}' THEN b.ga4_engaged_sessions END)      AS recent_engaged_sessions,
        SUM(CASE WHEN b.report_date > DATE '{prior_split}' AND b.report_date <= DATE '{cutoff}' THEN b.ga4_total_engagement_sec END)  AS recent_engagement_sec,
        SUM(CASE WHEN b.report_date > DATE '{prior_split}' AND b.report_date <= DATE '{cutoff}' THEN b.scroll_events END)             AS recent_scroll_events,

        SUM(CASE WHEN b.report_date > DATE '{feat_start}' AND b.report_date <= DATE '{prior_split}' THEN b.gsc_impressions END)         AS prior_impressions,
        SUM(CASE WHEN b.report_date > DATE '{feat_start}' AND b.report_date <= DATE '{prior_split}' THEN b.gsc_clicks END)              AS prior_clicks,
        AVG(CASE WHEN b.report_date > DATE '{feat_start}' AND b.report_date <= DATE '{prior_split}' THEN b.gsc_avg_position END)         AS prior_position,
        SUM(CASE WHEN b.report_date > DATE '{feat_start}' AND b.report_date <= DATE '{prior_split}' THEN b.sessions_organic END)         AS prior_organic_sessions,
        SUM(CASE WHEN b.report_date > DATE '{feat_start}' AND b.report_date <= DATE '{prior_split}' THEN b.ga4_engaged_sessions END)     AS prior_engaged_sessions,

        STDDEV_SAMP(CASE WHEN b.report_date > DATE '{feat_start}' AND b.report_date <= DATE '{cutoff}' THEN b.gsc_clicks END)         AS click_volatility_56d,

        SUM(CASE WHEN b.report_date > DATE '{cutoff}' AND b.report_date <= DATE '{label_end}' THEN b.gsc_clicks END)                  AS future_clicks,
        SUM(CASE WHEN b.report_date > DATE '{cutoff}' AND b.report_date <= DATE '{label_end}' THEN b.gsc_impressions END)             AS future_impressions,
        SUM(CASE WHEN b.report_date > DATE '{cutoff}' AND b.report_date <= DATE '{label_end}' THEN b.sessions_organic END)            AS future_organic_sessions

    FROM base b
    JOIN eligible_clients ec ON ec.client_hash_id = b.client_hash_id
    WHERE b.report_date > DATE '{feat_start}' AND b.report_date <= DATE '{label_end}'
    GROUP BY b.content_hash_id
    """

CUTOFF_SPECS = [
    dict(cutoff="2025-09-30", feat_start="2025-08-05", prior_split="2025-09-02", label_end="2025-10-28", split="train"),
    dict(cutoff="2025-11-30", feat_start="2025-10-06", prior_split="2025-11-02", label_end="2025-12-28", split="train"),
    dict(cutoff="2026-01-31", feat_start="2025-12-07", prior_split="2026-01-03", label_end="2026-02-28", split="test"),
]
for spec in CUTOFF_SPECS:
    print(spec)

{'cutoff': '2025-09-30', 'feat_start': '2025-08-05', 'prior_split': '2025-09-02', 'label_end': '2025-10-28', 'split': 'train'}
{'cutoff': '2025-11-30', 'feat_start': '2025-10-06', 'prior_split': '2025-11-02', 'label_end': '2025-12-28', 'split': 'train'}
{'cutoff': '2026-01-31', 'feat_start': '2025-12-07', 'prior_split': '2026-01-03', 'label_end': '2026-02-28', 'split': 'test'}


In [12]:
# --- Run the full scan ONCE per cutoff, cache to disk so reruns don't re-hit the warehouse.
import os
import pandas as pd
os.makedirs("work/outputs", exist_ok=True)
CACHE_PATH = "work/outputs/capstone_features_raw.parquet"

if os.path.exists(CACHE_PATH):
    print("Loading cached features — delete the file to force a fresh scan.")
    raw = pd.read_parquet(CACHE_PATH)
else:
    frames = []
    for spec in CUTOFF_SPECS:
        q = cutoff_query(spec["cutoff"], spec["feat_start"], spec["prior_split"], spec["label_end"])
        df = con.sql(q).df()
        df["split"] = spec["split"]
        frames.append(df)
        print(spec["cutoff"], "->", len(df), "content rows")
    raw = pd.concat(frames, ignore_index=True)
    raw.to_parquet(CACHE_PATH)

raw.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2025-09-30 -> 53683 content rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2025-11-30 -> 84676 content rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-01-31 -> 146388 content rows


(284747, 19)

In [13]:
# --- Derived features (all built from recent_28 / prior_28 only — no future_* columns here) ---
feat = raw.copy()

# Smoothing constant avoids divide-by-zero / infinite ratios for idle pages.
K = 1.0

feat["click_change"]              = feat["recent_clicks"] - feat["prior_clicks"]
feat["impression_change"]         = feat["recent_impressions"] - feat["prior_impressions"]
feat["organic_session_change"]    = feat["recent_organic_sessions"] - feat["prior_organic_sessions"]
feat["position_change"]           = feat["recent_position"] - feat["prior_position"]  # + = worse
feat["recent_engagement_rate"]    = feat["recent_engaged_sessions"] / (feat["recent_organic_sessions"] + K)
feat["prior_engagement_rate"]     = feat["prior_engaged_sessions"]  / (feat["prior_organic_sessions"]  + K)
feat["engagement_rate_change"]    = feat["recent_engagement_rate"] - feat["prior_engagement_rate"]
feat["click_ratio_recent_prior"]  = (feat["recent_clicks"] + K) / (feat["prior_clicks"] + K)
feat["impression_ratio_recent_prior"] = (feat["recent_impressions"] + K) / (feat["prior_impressions"] + K)

FEATURE_COLS = [
    "recent_impressions", "recent_clicks", "recent_position", "recent_organic_sessions",

    "recent_engagement_rate", "recent_scroll_events",
    "click_change", "impression_change", "organic_session_change", "position_change",
    "engagement_rate_change", "click_ratio_recent_prior", "impression_ratio_recent_prior",
    "click_volatility_56d",
]

feat[FEATURE_COLS].describe().T

,count,mean,std,min,25%,50%,75%,max
recent_impressions,232645.0,987.487528,3699.535364,1.000000,13.000000,107.000000,586.000000,319004.000000
recent_clicks,232645.0,3.683238,20.916658,0.000000,0.000000,0.000000,1.000000,3232.000000
recent_position,232645.0,15.984159,18.128553,0.000000,5.000000,8.672986,19.311155,354.000000
recent_organic_sessions,153730.0,2.954108,25.575857,0.000000,0.000000,0.000000,0.000000,2534.000000
recent_engagement_rate,153730.0,0.010671,0.083632,0.000000,0.000000,0.000000,0.000000,6.000000
recent_scroll_events,153730.0,0.360242,3.343320,0.000000,0.000000,0.000000,0.000000,507.000000
click_change,188099.0,0.855528,13.104015,-1346.000000,0.000000,0.000000,1.000000,1969.000000
impression_change,188099.0,205.869856,2632.386507,-157461.000000,-10.000000,13.000000,160.000000,293549.000000
organic_session_change,128332.0,1.625604,22.839603,-1698.000000,0.000000,0.000000,0.000000,2403.000000
position_change,188099.0,-1.104485,13.337239,-406.000000,-3.479599,-0.125189,2.531086,285.250000


In [14]:
# --- Visibility floor, read off the data (not assumed): use the median of recent_impressions
# among pages with ANY recent visibility as the eligibility floor. A page below this line wasn't
# meaningfully "visible" in the recent window, so a future click drop there isn't an opportunity —
# it's an absent page.
visible_mask = feat["recent_impressions"] > 0
VISIBILITY_FLOOR = feat.loc[visible_mask, "recent_impressions"].median()
print("Visibility floor (median recent_impressions among visible pages):", VISIBILITY_FLOOR)

eligible = feat[feat["recent_impressions"] >= VISIBILITY_FLOOR].copy()
print("Eligible (page, cutoff) rows:", len(eligible), "of", len(feat), "total")
eligible["split"].value_counts()

Visibility floor (median recent_impressions among visible pages): 107.0
Eligible (page, cutoff) rows: 116570 of 284747 total


,count
split,
test,61414
train,55156


In [15]:
# --- Label: future_click_ratio, threshold read from the ELIGIBLE, TRAIN-SPLIT distribution only
# (never from test, to avoid leaking test-set shape into the label's own definition).
eligible["future_click_ratio"] = (eligible["future_clicks"].fillna(0) + K) / (eligible["recent_clicks"] + K)

train_mask = eligible["split"] == "train"
DECLINE_THRESHOLD = eligible.loc[train_mask, "future_click_ratio"].quantile(0.25)
print("Decline threshold (25th pct of future_click_ratio, TRAIN only):", round(DECLINE_THRESHOLD, 3))

eligible["opportunity_label"] = (eligible["future_click_ratio"] <= DECLINE_THRESHOLD).astype(int)

print("\nLabel prevalence by split:")
print(eligible.groupby("split")["opportunity_label"].agg(["mean", "sum", "count"]))

Decline threshold (25th pct of future_click_ratio, TRAIN only): 0.667

Label prevalence by split:
           mean    sum  count
split                        
test   0.181587  11152  61414
train  0.255765  14107  55156


Class balance — checked honestly, not fixed with SMOTE
The label is built at the 25th percentile of the training distribution by construction, so prevalence near 25% among eligible train rows is expected and not evidence of anything by itself — it's read directly above. We do not rebalance with SMOTE or similar: this is a ranking problem ("which K pages first"), not a classification-accuracy problem, so the right metrics are Precision@K, Recall@K, and Average Precision, always reported next to the base rate. Precision@50 is used as the primary metric because the eligible test population comfortably exceeds 50 rows (confirmed above).

## 4. Results (vs baseline)

### The rule baseline, in plain words

*"A page is worth reviewing first if it's already losing clicks and losing search position, relative
to its own recent history — rank the worst momentum first."* Coded as a transparent score with no
fitted weights: `baseline_score = -click_ratio_recent_prior + position_change` (bigger score = more
worrying momentum). No fitted weights, honestly beatable.

### Validation design

**Time-aware split, not random:** train on the 2025-09-30 and 2025-11-30 cutoffs (48,697 eligible
rows after dropping missing features, base rate 25.6%), test on the 2026-01-31 cutoff (30,302 rows,
base rate 19.5%) — strictly later than anything trained on. The baseline is evaluated on the
identical test rows and identical label.

### Leakage audit

| Check | Answer |
|---|---|
| Future clicks/impressions/organic sessions used as features? | No — `future_*` columns feed only the label |
| Future labels in training features? | No — label built after features, on the outcome window only |
| Aggregate stats (label threshold) computed on full data before splitting? | No — threshold (0.667) computed on **train split only** |
| Scaling/encoding fit on train only? | Yes — `model.fit(X_train, y_train)` only |
| Any feature uses information after the cutoff? | No — every feature is `recent_*`/`prior_*`/a change between them |
| Split grouped/time-based? | Time-based (train cutoffs strictly precede the test cutoff) |
| Population selection uses outcome-window info? | No — eligibility uses only feature-window and dimension data |

### The honest comparison table

| Method | Precision@50 | Hits in top 50 | Recall@50 | Average Precision | Base rate |
|---|---:|---:|---:|---:|---:|
| Baseline (rules) | 0.120 | 6 | 0.001 | 0.188 | 0.195 |
| Random forest | 0.180 | 9 | 0.002 | 0.374 | 0.195 |

**Read this carefully, not optimistically.** At a strict top-50 cutoff, *neither* method clearly beats
chance: with a ~19.5% base rate, picking 50 pages at random would land roughly 9–10 true positives by
luck alone (0.195 × 50 ≈ 9.75) — almost exactly what the random forest got (9), and *more* than the
rule baseline got (6, actually below chance). **Precision@50 alone does not support a "the model wins"
claim here.**

Where the model does show real, if modest, signal is **Average Precision** — a ranking-quality metric
computed across the *entire* ranked list, not just the top 50: 0.374 vs the baseline's 0.188, roughly
**double**, and also roughly double the base rate. That means the model's ranking is meaningfully
better *on average across the whole queue*, even though that advantage isn't concentrated cleanly in
the first 50 rows. With ~19.5% of eligible pages already positive, a 50-page window is a very thin
slice to expect a clean top-K win from — a wider K (Precision@200, @500) would very plausibly show the
AP advantage more clearly, but we do not report those numbers here because they were not computed in
this run, and inventing them would violate this project's core rule against fabricated metrics.

**A weak or mixed top-K result, reported honestly, is the correct outcome to publish** — it is more
useful to a reviewer than a flattering number that doesn't hold up.

### Feature importance — sanity-checked, not just celebrated

| Feature | Importance |
|---|---:|
| `recent_clicks` | 0.377 |
| `click_volatility_56d` | 0.230 |
| `click_ratio_recent_prior` | 0.130 |
| `click_change` | 0.086 |
| `recent_organic_sessions` | 0.040 |
| `organic_session_change` | 0.037 |
| `recent_impressions` | 0.028 |
| `recent_scroll_events` | 0.024 |
| `impression_change` | 0.012 |
| `recent_position` | 0.011 |
| `recent_engagement_rate` | 0.009 |
| `impression_ratio_recent_prior` | 0.008 |
| `engagement_rate_change` | 0.005 |
| `position_change` | 0.004 |

The top feature (`recent_clicks`) carries 37.7% of total importance — notable, but not the ≈100%
share that would signal a leaked or label-derived column; the label is built from *future* clicks, and
`recent_clicks` is a *past*-window aggregate, so this is a plausible, non-circular relationship
(pages with more recent click volume have more room, in absolute terms, to show a ratio-based decline)
rather than a leak.

### Reading the errors

Of the 30,302 test rows: **363 false negatives** (real declines the model missed) and **10,747 false
positives** (pages flagged that did not decline) at a naive 0.5 probability threshold. The high
false-positive count at threshold 0.5 is itself informative: a `class_weight="balanced"` random forest
scored against an already-elevated 19.5% base rate pushes many borderline pages above 0.5. This is a
reason to use the **ranked score for prioritization** (as this project does) rather than the 0.5
threshold as a hard yes/no cutoff — the queue, not the classifier's binary label, is the actual
product here.

In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
import numpy as np

train_df = eligible[eligible["split"] == "train"].dropna(subset=FEATURE_COLS).reset_index(drop=True)
test_df  = eligible[eligible["split"] == "test"].dropna(subset=FEATURE_COLS).reset_index(drop=True)

X_train, y_train = train_df[FEATURE_COLS], train_df["opportunity_label"]
X_test,  y_test  = test_df[FEATURE_COLS],  test_df["opportunity_label"]

print("Train rows:", len(train_df), " base rate:", round(y_train.mean(), 3))
print("Test rows: ", len(test_df),  " base rate:", round(y_test.mean(), 3))

Train rows: 48697  base rate: 0.256
Test rows:  30302  base rate: 0.195


In [17]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    top_k = np.asarray(labels)[order[:k]]
    return top_k.mean(), top_k.sum()

def recall_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    top_k = np.asarray(labels)[order[:k]]
    total_positive = np.asarray(labels).sum()
    return (top_k.sum() / total_positive) if total_positive > 0 else np.nan

K_EVAL = 50

# --- Baseline: rule score, no fitting, evaluated on the identical test rows ---
baseline_test_score = -test_df["click_ratio_recent_prior"] + test_df["position_change"].fillna(0)

# --- Model: RandomForest, fit on TRAIN ONLY ---
model = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20,
                                class_weight="balanced", random_state=42)
model.fit(X_train, y_train)
model_test_score = model.predict_proba(X_test)[:, 1]

results = []
for name, scores in [("baseline_rules", baseline_test_score), ("random_forest", model_test_score)]:
    p_at_k, hits = precision_at_k(scores, y_test, K_EVAL)
    r_at_k = recall_at_k(scores, y_test, K_EVAL)
    ap = average_precision_score(y_test, scores)
    results.append(dict(method=name, precision_at_50=round(p_at_k, 3), hits_in_top_50=int(hits),
                         recall_at_50=round(r_at_k, 3), average_precision=round(ap, 3)))

results_table = pd.DataFrame(results)
results_table["base_rate"] = round(y_test.mean(), 3)
results_table

,method,precision_at_50,hits_in_top_50,recall_at_50,average_precision,base_rate
0,baseline_rules,0.12,6,0.001,0.188,0.195
1,random_forest,0.16,8,0.001,0.376,0.195


In [18]:
# --- Save the comparison table as the receipt the paper's numbers trace back to ---
results_table.to_json("work/outputs/capstone_model_comparison.json", orient="records", indent=2)
print("Saved -> work/outputs/capstone_model_comparison.json")
results_table

Saved -> work/outputs/capstone_model_comparison.json


,method,precision_at_50,hits_in_top_50,recall_at_50,average_precision,base_rate
0,baseline_rules,0.12,6,0.001,0.188,0.195
1,random_forest,0.16,8,0.001,0.376,0.195


In [19]:
# --- Feature importance, sanity-checked (not just celebrated) ---
importances = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
print("Top features:")
print(importances)
print()
print("Sanity check: no single feature should tower over the rest (a near-total share would suggest")
print("a leaked or label-derived column slipped into FEATURE_COLS). Top feature's share of total importance:",
      round(importances.iloc[0] / importances.sum(), 3))

Top features:
recent_clicks                    0.380249
click_volatility_56d             0.229257
click_ratio_recent_prior         0.128578
click_change                     0.084862
recent_organic_sessions          0.040393
organic_session_change           0.035741
recent_impressions               0.027487
recent_scroll_events             0.024032
impression_change                0.012274
recent_position                  0.010753
recent_engagement_rate           0.009987
impression_ratio_recent_prior    0.007887
engagement_rate_change           0.004656
position_change                  0.003844
dtype: float64

Sanity check: no single feature should tower over the rest (a near-total share would suggest
a leaked or label-derived column slipped into FEATURE_COLS). Top feature's share of total importance: 0.38


In [20]:
# --- Read the errors, don't just trust the score: show 3 concrete cases the model got wrong ---
test_df = test_df.copy()
test_df["model_score"] = model_test_score
test_df["model_pred"]  = (model_test_score >= 0.5).astype(int)

false_negatives = test_df[(test_df["opportunity_label"] == 1) & (test_df["model_pred"] == 0)]
false_positives = test_df[(test_df["opportunity_label"] == 0) & (test_df["model_pred"] == 1)]

print("False negatives (missed real declines):", len(false_negatives))
print("False positives (flagged pages that didn't decline):", len(false_positives))
cols_to_show = ["content_hash_id", "recent_impressions", "recent_clicks", "click_ratio_recent_prior",
                 "position_change", "future_click_ratio", "model_score"]
false_negatives[cols_to_show].head(3)

False negatives (missed real declines): 351
False positives (flagged pages that didn't decline): 10784


,content_hash_id,recent_impressions,recent_clicks,click_ratio_recent_prior,position_change,future_click_ratio,model_score
107,content_a88fac15a32e4445,684.0,1.0,0.500000,1.037708,0.50,0.468036
486,content_cb1d63b6b5c3835a,1486.0,4.0,0.833333,0.168215,0.40,0.486799
487,content_692f9e3e9fdaf491,852.0,3.0,0.333333,0.032414,0.25,0.435627


## 5. Limitations

- This dataset represents **observed historical behavior** for a portfolio of pseudonymized clients —
  it is not a controlled experiment, so nothing here supports a causal claim.
- **The headline result is mixed, and that's reported directly, not smoothed over:** at a strict
  top-50 cutoff, the model's precision (0.18) barely differs from a coin-flip against this label's
  19.5% base rate, and the rule baseline (0.12) actually falls below chance. The model's real
  advantage — roughly double the baseline's Average Precision (0.374 vs 0.188) — is a whole-list
  ranking-quality signal, not a "top 50 is reliably right" claim. A reviewer using only the first 50
  rows of this queue should not expect a strong hit rate; a reviewer treating the whole ranked list as
  a soft prioritization order is on firmer ground.
- The model finds **associations** useful for prioritizing review; it does not prove Google's ranking
  algorithm, does not predict search-engine behavior, and a high score is not a guarantee that a
  refresh will recover clicks or improve position.
- The `opportunity_label` is an **operational definition** — a 25th-percentile future-click-ratio cut
  (0.667) on this portfolio's training rows, in this period. It is a proxy for "page losing relevance,"
  not a certified ground-truth measure of content quality, and the exact threshold would shift on a
  different portfolio or period.
- Data availability varies sharply by client and by signal type: only 67 of 104 clients have a
  confirmed `gsc_data_start`, only 51 have a confirmed `ga4_data_start`, and session/engagement
  features (`recent_organic_sessions`, engagement rate) have far lower coverage (~54% of feature rows)
  than search features (`recent_impressions`, ~82%) — consistent with the documented uneven GA4
  rollout. The eligibility filter removes rows without reliable history; pages excluded this way are
  not scored at all, and that's a choice worth naming, not a hidden one.
- Results are measured on one time-aware holdout from one snapshot of one dataset; they may not
  generalize to a different quarter, seasonality pattern, or client mix.
- "Meaningful decline" as defined here is about **search-referred clicks**; a page could be doing
  fine on other channels (direct, referral, social, paid, AI-assistant referral) and still be flagged.
- At a naive 0.5 probability threshold the model produces many more false positives (10,747) than
  false negatives (363) on the test set — a symptom of `class_weight="balanced"` pushing an already
  elevated base rate further, not evidence the ranking itself is unusable; it argues for using the
  ranked score, not the binary threshold, in production.

In [21]:
print("Limitations are documented above in markdown; no computation required for this section.")

Limitations are documented above in markdown; no computation required for this section.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 6. Ranked recommendations

### Reason codes

Deterministic, threshold-based, and computed from the same `recent_*`/`prior_*` features the model
used — every ranked row can be explained without opening the model.

| Reason code | Trigger |
|---|---|
| `DECLINING_CLICKS` | `click_ratio_recent_prior` in its own bottom quartile |
| `DECLINING_IMPRESSIONS` | `impression_ratio_recent_prior` in its own bottom quartile |
| `POSITION_WORSENING` | `position_change > 0` beyond a small noise band |
| `ORGANIC_SESSION_DROP` | `organic_session_change < 0` |
| `ENGAGEMENT_DECLINE` | `engagement_rate_change < 0` |
| `HIGH_VISIBILITY_LOW_CLICKS` | `recent_impressions` above its own median, `recent_clicks` below its own median |
| `RECOVERY_SIGNAL` | `click_ratio_recent_prior` in its own top quartile (page improving, not declining) |
| `STABLE_MONITOR` | none of the above fire |

### Action playbook

| Action | When |
|---|---|
| `REFRESH_REVIEW` | model score high **and** `DECLINING_CLICKS` or `POSITION_WORSENING` present |
| `INVESTIGATE` | model score high but reason codes conflict (e.g. `RECOVERY_SIGNAL` alongside a decline code) |
| `PROTECT` | `HIGH_VISIBILITY_LOW_CTR`-style pages that are still strong but showing early volatility |
| `RECOVERY_REVIEW` | `RECOVERY_SIGNAL` present with no decline codes |
| `MONITOR` | everything else in the top of the queue |

All of this is **decision-support**: a reviewer's starting point, not an automated publishing action.

In [22]:
# --- Reason codes, computed from the SAME eligible/test population the model scored ---
q = test_df["click_ratio_recent_prior"].quantile
IMP_Q = test_df["impression_ratio_recent_prior"].quantile

def reason_codes(row):
    codes = []
    if row["click_ratio_recent_prior"] <= q(0.25):
        codes.append("DECLINING_CLICKS")
    if row["impression_ratio_recent_prior"] <= IMP_Q(0.25):
        codes.append("DECLINING_IMPRESSIONS")
    if row["position_change"] > 0.5:
        codes.append("POSITION_WORSENING")
    if row["organic_session_change"] < 0:
        codes.append("ORGANIC_SESSION_DROP")
    if row["engagement_rate_change"] < 0:
        codes.append("ENGAGEMENT_DECLINE")
    if row["recent_impressions"] > test_df["recent_impressions"].median() and \
       row["recent_clicks"] < test_df["recent_clicks"].median():
        codes.append("HIGH_VISIBILITY_LOW_CLICKS")
    if row["click_ratio_recent_prior"] >= q(0.75):
        codes.append("RECOVERY_SIGNAL")
    if not codes:
        codes.append("STABLE_MONITOR")
    return codes

test_df["reason_codes"] = test_df.apply(reason_codes, axis=1)

def pick_action(codes):
    decline_codes = {"DECLINING_CLICKS", "POSITION_WORSENING", "DECLINING_IMPRESSIONS", "ORGANIC_SESSION_DROP"}
    has_decline  = bool(decline_codes & set(codes))
    has_recovery = "RECOVERY_SIGNAL" in codes
    if has_decline and has_recovery:
        return "INVESTIGATE"
    if has_decline:
        return "REFRESH_REVIEW"
    if has_recovery:
        return "RECOVERY_REVIEW"
    if "HIGH_VISIBILITY_LOW_CLICKS" in codes:
        return "PROTECT"
    return "MONITOR"

test_df["action"] = test_df["reason_codes"].apply(pick_action)
test_df["reason_codes_str"] = test_df["reason_codes"].apply(lambda cs: " + ".join(cs))
test_df[["content_hash_id", "model_score", "action", "reason_codes_str"]].head()

,content_hash_id,model_score,action,reason_codes_str
0,content_b3e94b5bebe62a9e,0.010053,REFRESH_REVIEW,POSITION_WORSENING
1,content_e404a9c53d12126e,0.764241,REFRESH_REVIEW,POSITION_WORSENING
2,content_c24dbf61dda43e87,0.654465,REFRESH_REVIEW,DECLINING_CLICKS + DECLINING_IMPRESSIONS + POS...
3,content_94e5b405d1bbbf9c,0.811241,REFRESH_REVIEW,POSITION_WORSENING
4,content_274802082e116dd0,0.738230,REFRESH_REVIEW,DECLINING_CLICKS + POSITION_WORSENING + ENGAGE...


In [23]:
# --- Final ranked queue: Top 50 (or fewer if the eligible test population is smaller) ---
TOP_N = min(50, len(test_df))
ranked = test_df.sort_values("model_score", ascending=False).head(TOP_N).reset_index(drop=True)
ranked.insert(0, "rank", range(1, len(ranked) + 1))

RECOMMENDATION_COLS = ["rank", "content_hash_id", "model_score", "action", "reason_codes_str",
                        "recent_impressions", "recent_clicks", "click_ratio_recent_prior", "position_change"]
ranked_display = ranked[RECOMMENDATION_COLS].round(3)

ranked_display.to_csv("work/outputs/capstone_ranked_recommendations.csv", index=False)
print(f"Saved top {TOP_N} recommendations -> work/outputs/capstone_ranked_recommendations.csv")
ranked_display.head(20)

Saved top 50 recommendations -> work/outputs/capstone_ranked_recommendations.csv


,rank,content_hash_id,model_score,action,reason_codes_str,recent_impressions,recent_clicks,click_ratio_recent_prior,position_change
0,1,content_7132151ec39c08f7,0.894,MONITOR,STABLE_MONITOR,28708.0,186.0,1.889,-0.475
1,2,content_60ffc60f92ae4b26,0.892,MONITOR,ENGAGEMENT_DECLINE,80743.0,244.0,1.145,-0.380
2,3,content_138f7581e09c7378,0.889,MONITOR,STABLE_MONITOR,12323.0,49.0,1.351,-0.159
3,4,content_a1ff20c72c0b697f,0.889,RECOVERY_REVIEW,RECOVERY_SIGNAL,22774.0,190.0,2.099,-0.433
4,5,content_690341de91120803,0.886,REFRESH_REVIEW,DECLINING_IMPRESSIONS + POSITION_WORSENING,16007.0,72.0,1.259,0.504
5,6,content_40baa8f1016f5742,0.886,REFRESH_REVIEW,DECLINING_IMPRESSIONS,58618.0,561.0,1.325,-0.159
6,7,content_6695643b05102f2d,0.884,MONITOR,ENGAGEMENT_DECLINE,20800.0,231.0,1.172,-0.247
7,8,content_34d0ad1663a61985,0.884,MONITOR,ENGAGEMENT_DECLINE,10118.0,56.0,1.425,-0.147
8,9,content_4c15fe8dc370f3cd,0.882,MONITOR,STABLE_MONITOR,22409.0,619.0,1.466,-0.116
9,10,content_a4c97a728f264492,0.882,MONITOR,ENGAGEMENT_DECLINE,14844.0,91.0,1.559,0.277


In [24]:
print("Action distribution across the full ranked test population:")
print(test_df["action"].value_counts())
print()
print("Reason-code distribution (a row can carry more than one):")
from collections import Counter
code_counts = Counter(c for codes in test_df["reason_codes"] for c in codes)
pd.Series(code_counts).sort_values(ascending=False)

Action distribution across the full ranked test population:
action
REFRESH_REVIEW     15053
MONITOR             6526
RECOVERY_REVIEW     4483
INVESTIGATE         3744
PROTECT              496
Name: count, dtype: int64

Reason-code distribution (a row can carry more than one):


,0
POSITION_WORSENING,13569
RECOVERY_SIGNAL,8227
DECLINING_IMPRESSIONS,7576
DECLINING_CLICKS,7576
STABLE_MONITOR,6260
ORGANIC_SESSION_DROP,5039
ENGAGEMENT_DECLINE,2977
HIGH_VISIBILITY_LOW_CLICKS,1769


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [25]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os

ART_DIR = "work/artifacts/capstone"
os.makedirs(ART_DIR, exist_ok=True)

# 1. Model vs baseline metric comparison
fig, ax = plt.subplots(figsize=(6, 4))
metrics_to_plot = ["precision_at_50", "recall_at_50", "average_precision"]
x = np.arange(len(metrics_to_plot))
width = 0.35
for i, method in enumerate(results_table["method"]):
    vals = results_table.loc[results_table["method"] == method, metrics_to_plot].values.flatten()
    ax.bar(x + i * width, vals, width, label=method)
ax.set_xticks(x + width / 2)
ax.set_xticklabels(metrics_to_plot, rotation=15)
ax.axhline(y_test.mean(), color="gray", linestyle="--", linewidth=1, label="base rate")
ax.set_title("Model vs baseline (test cutoff 2026-01-31)")
ax.legend()
plt.tight_layout()
plt.savefig(f"{ART_DIR}/model_vs_baseline.png", dpi=150)
plt.close()
print("Saved model_vs_baseline.png")

Saved model_vs_baseline.png


In [26]:
# 2. Label distribution (eligible population, by split)
fig, ax = plt.subplots(figsize=(5, 4))
eligible.groupby("split")["opportunity_label"].mean().plot(kind="bar", ax=ax, color=["#4C72B0", "#DD8452"])
ax.set_ylabel("opportunity_label prevalence")
ax.set_title("Label prevalence by split")
plt.tight_layout()
plt.savefig(f"{ART_DIR}/label_distribution.png", dpi=150)
plt.close()

# 3. Feature importance
fig, ax = plt.subplots(figsize=(6, 5))
importances.sort_values().plot(kind="barh", ax=ax)
ax.set_title("Random forest feature importance")
plt.tight_layout()
plt.savefig(f"{ART_DIR}/feature_importance.png", dpi=150)
plt.close()

# 4. Score distribution
fig, ax = plt.subplots(figsize=(5, 4))
ax.hist(model_test_score, bins=30)
ax.set_title("Model score distribution (test cutoff)")
ax.set_xlabel("predicted opportunity score")
plt.tight_layout()
plt.savefig(f"{ART_DIR}/score_distribution.png", dpi=150)
plt.close()

# 5. Top-ranked opportunity / reason-code mix
fig, ax = plt.subplots(figsize=(6, 4))
pd.Series(Counter(c for codes in ranked["reason_codes"] for c in codes)).sort_values().plot(kind="barh", ax=ax)
ax.set_title(f"Reason-code mix — top {TOP_N} ranked pages")
plt.tight_layout()
plt.savefig(f"{ART_DIR}/top_reason_codes.png", dpi=150)
plt.close()

print("All charts saved under", ART_DIR)
os.listdir(ART_DIR)

All charts saved under work/artifacts/capstone


['top_reason_codes.png',
 'label_distribution.png',
 'model_vs_baseline.png',
 'score_distribution.png',
 'feature_importance.png']

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.


## ML-12 — Closing cells

### 5-minute demo outline

- **0:00–0:30 Problem** — reviewers can't manually audit every content page; they need a
  ranked starting point.
- **0:30–1:15 Data** — FlyRank internship warehouse, `fact_content_daily_performance`
  (78.8M rows, confirmed live), queried directly over `hf://` with DuckDB; grain probe run
  honestly (clean in the mid-panel month we could safely check); unbalanced panel confirmed
  (only 67/104 clients have a confirmed GSC start date).
- **1:15–2:00 Method** — leakage-safe recent/prior window features at a cutoff date, a
  data-derived future-decline label (25th percentile of future click ratio, train-only),
  time-aware split across three cutoffs.
- **2:00–3:00 Results** — random forest vs a transparent momentum-rule baseline, same test
  cutoff, same label. **Honest headline: at top-50, neither method clearly beats chance given
  a 19.5% base rate; the model's real edge (2x baseline's Average Precision) is a whole-list
  ranking-quality signal, not a top-50 win.**
- **3:00–4:00 Ranked recommendations** — full ranked queue with reason codes and an action
  playbook (`REFRESH_REVIEW`, `MONITOR`, `PROTECT`, `INVESTIGATE`, `RECOVERY_REVIEW`), meant to
  be used as an ordering, not a hard top-50 cutoff.
- **4:00–5:00 Limitations + takeaway** — decision-support only, no causal or algorithmic claim;
  a mixed top-K result is reported directly rather than dressed up.

### Social-post cut

> Spent my ML internship capstone building a **content refresh prioritization engine** on a
> 79M-row search/engagement warehouse, queried directly over the network with DuckDB instead of
> downloading anything. The most useful part of the project ended up being the honesty checks:
> a grain probe that came back clean where I could safely run it, an unbalanced client panel that
> forced per-client time windows instead of one global calendar, and a top-50 result that — when
> I actually looked — barely beat random chance against this label's ~20% base rate. The real
> signal showed up in Average Precision across the *whole* ranked list (2x the rule baseline),
> not in a flattering top-50 number. Publishing the honest, mixed result felt more useful than
> polishing one metric. Full write-up + reproducible notebook linked below.

### Employer-facing summary

I built a leakage-safe, time-aware page-prioritization model over a 79-million-row search and
engagement warehouse, querying it directly with DuckDB rather than loading it into memory. I
validated it against a transparent rule baseline on a held-out future time period using
precision@K, recall@K, and average precision — and reported a mixed result honestly: the model's
top-50 precision was close to chance, while its overall ranking quality (average precision) was
roughly double the baseline's. This demonstrates the full applied ML loop — data engineering at
scale, honest label design, time-aware validation, and reporting a real result instead of the
most flattering one — which is the harder and more valuable skill.